# Experiment 3: CPU Colab notebook
## Restricted-population workload-disjoint robustness — validation only

Historical test workloads from **both** historical partition seeds have been excluded using their saved ID-only split maps. This is **not a direct full-population replication**.

| Population | Executions | Workloads |
|---|---:|---:|
| Original | 4,709,624 | 473,265 |
| Excluded historical holdout union | 1,337,194 | 131,648 |
| Eligible | 3,372,430 | 341,617 |

Upload the accompanying **Experiment3_bundle.zip** to Drive. This notebook runs CPU LightGBM 4.7.0 with the original feature processing, objective and twelve-candidate grid. Checkpoints go directly to Drive. Only one seed runs at a time.

**Run cells in order.** The first seed is a complete 24-fit memory/runtime pilot. The remaining nine seeds are gated on pilot success. No research fit has been run or claimed in this delivered notebook. Neither historical nor new confirmatory test outcomes are evaluated.


## 1. Install pinned packages
Use a CPU runtime with about 12 GB RAM. Dependencies are imported only inside fresh subprocesses after installation, so existing notebook-kernel imports do not silently supply a different version. Do not change package versions after registration. Installation and Drive access require internet connectivity in Colab.

In [ ]:
import subprocess, sys
requirements = ["lightgbm==4.7.0", "numpy==2.0.1", "pandas==2.2.3", "pyarrow==25.0.1", "psutil==6.1.1"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *requirements], check=True)
subprocess.run([sys.executable, "-c", "import lightgbm,numpy,pandas,pyarrow,psutil; print('LightGBM',lightgbm.__version__); assert lightgbm.__version__=='4.7.0'"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-c', "import lightgbm,numpy,pandas,pyarrow,psutil; print('LightGBM',lightgbm.__version__); assert lightgbm.__version__=='4.7.0'"], returncode=0)

## 2. Mount Drive and choose paths
Upload the bundle first. Edit these paths if your Drive folder differs. Use an empty `RUN_DIR` for the first run; reuse that exact directory for resumption. Run only one notebook against a run directory. No historical archive, test file or test-ID list is needed in Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ZIP_PATH = Path('/content/Experiment3_bundle.zip')

# Persistent checkpoints saved to Google Drive
RUN_DIR = Path('/content/drive/MyDrive/ASI_Experiment3/runs/exp3_restricted_v1')

Mounted at /content/drive


## 3. Verify and extract the eligible-only bundle locally
The archive checksum is fixed in this notebook. Membership maps containing excluded IDs are not deployed. Input reads use local Colab storage, while every completed scientific candidate is written and checksum-verified on Drive.

In [ ]:
import hashlib, json, shutil, tempfile, zipfile
EXPECTED_ZIP_SHA256 = "f102a8de54e37fbd5e065135f6717e2a4e7489b170117fdf8e683f38279e5ea0"
def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024**2), b''):
            h.update(block)
    return h.hexdigest()
assert ZIP_PATH.is_file(), f'Upload the supplied bundle to {ZIP_PATH}'
LOCAL = Path(tempfile.mkdtemp(prefix='asi_exp3_', dir='/content'))
LOCAL_ZIP = LOCAL / 'bundle.zip'
shutil.copyfile(ZIP_PATH, LOCAL_ZIP)
assert file_sha(LOCAL_ZIP) == EXPECTED_ZIP_SHA256, 'Wrong or damaged bundle'
PACKAGE = LOCAL / 'package'
PACKAGE.mkdir()
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    for member in archive.infolist():
        target = (PACKAGE / member.filename).resolve()
        assert target.is_relative_to(PACKAGE.resolve()), 'Unsafe archive member'
    archive.extractall(PACKAGE)
manifest = json.loads((PACKAGE / 'PACKAGE_MANIFEST.json').read_text())
for name, expected in manifest.items():
    assert file_sha(PACKAGE / name) == expected, f'Package mismatch: {name}'
BUNDLE = PACKAGE / 'inputs'
RUNNER = PACKAGE / 'exp3_runner.py'
print(json.dumps(json.loads((BUNDLE / 'population_audit.json').read_text()), indent=2))

def run_phase(phase, *extra):
    command = [sys.executable, '-u', str(RUNNER), phase,
               '--bundle', str(BUNDLE), '--out', str(RUN_DIR), *map(str, extra)]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as worker:
        for line in worker.stdout:
            print(line, end='', flush=True)
        status = worker.wait()
    if status:
        raise RuntimeError(f'{phase} stopped with exit status {status}; inspect output and Drive failure records')


{
  "historical_test_workloads": [
    70705,
    71500
  ],
  "overlap_workloads": 10557,
  "excluded_union_workloads": 131648,
  "original_workloads": 473265,
  "eligible_workloads": 341617,
  "historical_partition_seeds": [
    20260927,
    20261020
  ],
  "original_executions": 4709624,
  "eligible_executions": 3372430,
  "excluded_union_executions": 1337194,
  "original_population_source": "saved archival metadata: 4,709,624 rows / 473,265 workloads",
  "eligible_workload_fraction": 0.7218302642282864,
  "eligible_execution_fraction": 0.7160720261320225,
  "label": "new restricted-population robustness study; not direct full-population replication",
  "source_train_validation_counts": [
    {
      "path": "/Users/tiloschan/Documents/Codex/2026-10-26/referenced-chatgpt-conversation-this-is-an-5/outputs/ASI_recovery_20260927/train_20260927.parquet",
      "rows": 3339542,
      "eligible_rows": 2822729,
      "excluded_rows": 516813
    },
    {
      "path": "/Users/tiloschan/Doc

## 4. Freeze the preregistration before training
Ten new **split seeds**: 20261028, 20261029, 20261030, 20261031, 20261101, 20261102, 20261103, 20261104, 20261105, 20261106. Model seed remains **20260927** to isolate split sensitivity, as in experiment 2. These are ten split repetitions conditional on one model seed.

Workloads use the archived SHA-256 JSON assignment with 70/15/15 thresholds. M0 is R+P+A; M1 adds job_type_public. Train-only category encoding, regression_l1, 200-round cap, patience 30, the original twelve candidates and lowest-ID tie breaking are unchanged. Selection uses execution-weighted validation LogMAE; equal-workload LogMAE is secondary. New test membership is counted using identities only; test outcomes are not loaded.

The protocol, source, package versions and input hashes are bound to Drive checkpoints. Resume rejects changed data, code, parameters or package versions.

In [ ]:
run_phase('register')
registered = json.loads((RUN_DIR / 'PREREGISTRATION' / 'result.json').read_text())
SPEC = registered['spec']
assert len(SPEC['split_seeds']) == 10
print('Registration saved before fits:', RUN_DIR / 'PREREGISTRATION')
print('Split seeds:', SPEC['split_seeds'])
print('Model seed:', SPEC['model_seed'])


Registration saved before fits: /content/drive/MyDrive/ASI_Experiment3/runs/exp3_restricted_v1/PREREGISTRATION
Split seeds: [20261028, 20261029, 20261030, 20261031, 20261101, 20261102, 20261103, 20261104, 20261105, 20261106]
Model seed: 20260927


## 5. Run the one-seed memory/runtime pilot first
This executes all 12 M0 and all 12 M1 candidates for the first registered seed. Each candidate is checkpointed; it is part of the final ten-seed study. Resource checks stop rather than change the grid or dataset. No seed may be replaced because its result looks unusual.

The gate requires sampled peak process RSS ≤ 8 GiB and available system RAM ≥ 2 GiB. Sampling every 0.25 seconds plus per-round checks can miss brief peaks; the pilot is empirical evidence, not a guaranteed limit. Model quality does not determine pilot success.

In [ ]:
try:
    run_phase('seed', '--seed', SPEC['pilot_seed'])
finally:
    run_phase('report')
pilot = json.loads((RUN_DIR / 'PILOT' / 'result.json').read_text())
print(json.dumps(pilot, indent=2))
assert pilot['passed'], 'Pilot did not pass: remaining seeds must not run'
print('Estimated total candidate compute hours:', pilot['estimated_total_candidate_compute_seconds'] / 3600)
print('This estimate excludes loading, diagnostics, Drive and scheduling overhead.')


{"seed": 20261028, "role": "m0", "candidate": 1, "validation_logmae": 0.7307251348120064, "rounds": 200}
{"seed": 20261028, "role": "m0", "candidate": 2, "validation_logmae": 0.7337472338261365, "rounds": 106}
{"seed": 20261028, "role": "m0", "candidate": 3, "validation_logmae": 0.7307460163435243, "rounds": 199}
{"seed": 20261028, "role": "m0", "candidate": 4, "validation_logmae": 0.7337472338261365, "rounds": 106}
{"seed": 20261028, "role": "m0", "candidate": 5, "validation_logmae": 0.7266191232012559, "rounds": 177}
{"seed": 20261028, "role": "m0", "candidate": 6, "validation_logmae": 0.7269931096056376, "rounds": 98}
{"seed": 20261028, "role": "m0", "candidate": 7, "validation_logmae": 0.7266534378090295, "rounds": 156}
{"seed": 20261028, "role": "m0", "candidate": 8, "validation_logmae": 0.7268630343502009, "rounds": 114}
{"seed": 20261028, "role": "m0", "candidate": 9, "validation_logmae": 0.7273863115610882, "rounds": 105}
{"seed": 20261028, "role": "m0", "candidate": 10, "valid

## 6. Run the remaining nine seeds after reviewing pilot resources/time
Set `RUN_REMAINING = True` when you are ready. This does not alter the preregistration. If time is insufficient, stop and resume the same run later; committed candidates are reused. A failed or interrupted seed is not silently replaced or skipped. The runner checks the pilot gate even if invoked outside this notebook. The same physical-RAM class is required on resume; a changed RAM allocation requires a new run and pilot.

After disconnecting: mount Drive and rerun setup/registration/pilot against the same `RUN_DIR`, then this cell. Incomplete candidates are retained with `.incomplete-*` names and retrained. Committed checksum mismatches stop execution. Avoid simultaneous notebook instances. Drive writes are read back and checked; verify on resume because sudden disconnection may precede cloud synchronization.

In [ ]:
RUN_REMAINING = True
if RUN_REMAINING:
    try:
        for split_seed in SPEC['split_seeds']:
            if split_seed == SPEC['pilot_seed']:
                continue
            run_phase('seed', '--seed', split_seed)
    finally:
        run_phase('report')
else:
    print('Remaining seeds have not been launched. Inspect the pilot values above, then set RUN_REMAINING = True.')


 split_seed   status  elapsed_seconds  m0_best_iteration  m0_candidate_id  m0_equal_workload_logmae  m0_execution_logmae  m0_rows  m0_workloads  m1_best_iteration  m1_candidate_id  m1_equal_workload_logmae  m1_execution_logmae  m1_rows  m1_workloads  equal_workload_logmae_m0_minus_m1  equal_workload_logmae_relative_improvement_percent  execution_logmae_m0_minus_m1  execution_logmae_relative_improvement_percent
   20261028 complete      1433.662396                 58               12                  0.771419             0.726590   498078         51189                132                7                  0.753788             0.707546   498078         51189                           0.017631                                            2.285490                      0.019044                                       2.621041
   20261029 complete       962.608319                149                6                  0.774018             0.760963   513432         51328                 42          

## 7. Report all seed outcomes
Run this cell after any interruption or at completion. CSV lists all ten seeds, including partial/failed/not-started outcomes; JSON includes candidate records, rounds, diagnostics and failure history. An incomplete sweep is not a ten-seed result.

**Every model score is VALIDATION ONLY**, measured on data used for selection. LogMAE is absolute error on log1p(hours), not MAE in hours. Equal-workload LogMAE averages within each workload before averaging across workloads. Positive M0−M1 loss differences favor M1. No clipping is applied. Diagnostics include concentration, profile support, category support and error contributions.

New confirmatory test partitions and excluded historical holdouts remain untouched. Repeated splits overlap and are dependent: the across-seed summaries are descriptive, not independent-replication p-values. No confirmatory conclusion, causality claim or scheduler benefit follows from validation improvements.

In [ ]:
run_phase('report')
reports = sorted((RUN_DIR / 'reports').glob('report_*.json'), key=lambda p: p.name)
latest = json.loads(reports[-1].read_text())
print('All ten complete:', latest['all_ten_complete'])
print('Statuses:', latest['seed_status_counts'])
print('Confirmatory tests:', latest['confirmatory_tests'])
for key in ['execution_logmae_seed_summary', 'equal_workload_logmae_seed_summary']:
    if key in latest:
        print(key, json.dumps(latest[key], indent=2))
print('Full report:', reports[-1])
print('Per-seed diagnostics/models:', RUN_DIR / 'seeds')


 split_seed   status  elapsed_seconds  m0_best_iteration  m0_candidate_id  m0_equal_workload_logmae  m0_execution_logmae  m0_rows  m0_workloads  m1_best_iteration  m1_candidate_id  m1_equal_workload_logmae  m1_execution_logmae  m1_rows  m1_workloads  equal_workload_logmae_m0_minus_m1  equal_workload_logmae_relative_improvement_percent  execution_logmae_m0_minus_m1  execution_logmae_relative_improvement_percent
   20261028 complete      1433.662396                 58               12                  0.771419             0.726590   498078         51189                132                7                  0.753788             0.707546   498078         51189                           0.017631                                            2.285490                      0.019044                                       2.621041
   20261029 complete       962.608319                149                6                  0.774018             0.760963   513432         51328                 42          

## Files and provenance
`PREREGISTRATION/`: frozen study definition. `seeds/SEED/ROLE/candidate_NN/`: model, timing, score, selected rounds, resources and checksums. `SELECTION/`: all candidate records, winner and encoding. `SUMMARY/`: both validation metrics, concentration/profile-support diagnostics and eligible validation workload loss sums. `PILOT/`: resource gate/time. `reports/`: immutable timestamped snapshots. `FAILURE_*.json`: preserved failures.

The package includes `README.md`, population/input audit, archived protocol, source checksums and original source text. Archived source text is evidence; only the extracted feature/split functions and the new CPU runner are executed. Historical results and artifacts are not edited.


In [ ]:

from pathlib import Path
import pandas as pd

BASE = Path(
    "/content/drive/MyDrive/ASI_Experiment3/"
    "runs/exp3_restricted_v1/seeds"
)

seeds = [20261028, 20261029, 20261030, 20261031,
         20261101, 20261102, 20261103, 20261104,
         20261105, 20261106]

tables = {}

for seed in seeds:
    path = BASE / str(seed) / "SUMMARY" / "validation_workload_loss_sums.parquet"
    assert path.exists(), f"Missing file: {path}"
    tables[seed] = pd.read_parquet(path)

print("All ten workload-level files loaded.")
print("\nColumns:", tables[20261101].columns.tolist())
print("\nShape:", tables[20261101].shape)
print("\nData types:")
print(tables[20261101].dtypes)
print("\nSample rows:")
display(tables[20261101].head(5))


All ten workload-level files loaded.

Columns: ['workload_id', 'executions', 'm0_loss_sum', 'm1_loss_sum']

Shape: (50814, 4)

Data types:
workload_id     object
executions       int64
m0_loss_sum    float64
m1_loss_sum    float64
dtype: object

Sample rows:


,workload_id,executions,m0_loss_sum,m1_loss_sum
0,7c421c49c633914d9f7b7e572d73d711,179,332.217924,755.609431
1,c3a04fb0cf5d8e4fe091e9f76725aa52,14,31.661346,59.618475
2,38c2ba3b6eb3918dbee7fde9b708fc68,389,88.208894,90.836135
3,d51ec7cc6a40a1e2511251e7e94022e1,2807,1030.546255,1027.995316
4,ecc62dbdbeba7c289d59312a4e1c2c5c,4,9.728275,17.481323


RELATIVE M1 IMPROVEMENT (%)


scheme,cap_100,cap_1000,equal_workload,execution
seed,,,,
20261028,2.215,1.580,2.285,2.621
20261029,1.904,1.256,3.771,0.952
20261030,3.307,2.409,3.392,1.799
20261031,2.866,1.667,4.286,1.381
20261101,-9.380,-11.316,-2.457,-6.119
20261102,-2.211,-2.934,0.978,-2.015
20261103,3.248,2.463,3.515,2.099
20261104,2.640,2.076,2.925,1.369
20261105,2.610,1.846,3.723,1.167



TOP 1% FREQUENCY ATTRIBUTION


,seed,top1_exec_share_pct,top1_loss_diff_sum,rest_loss_diff_sum,total_loss_diff_sum,top1_delta_per_execution,rest_delta_per_execution
0,20261028,64.0103,5520.8940,3964.6166,9485.5106,0.0173,0.0221
1,20261029,65.0992,-50.8568,3771.8344,3720.9775,-0.0002,0.0210
2,20261030,63.0603,1104.5753,5743.2004,6847.7757,0.0035,0.0313
3,20261031,62.6342,-208.1109,5009.0202,4800.9093,-0.0007,0.0281
4,20261101,67.6080,-11929.7513,-12459.7044,-24389.4556,-0.0320,-0.0698
5,20261102,67.7037,-6518.5124,-2421.5439,-8940.0563,-0.0174,-0.0136
6,20261103,58.5194,2190.7979,5089.4712,7280.2691,0.0088,0.0289
7,20261104,65.6532,898.4321,4191.4223,5089.8544,0.0026,0.0229
8,20261105,64.0088,-333.7606,4345.9716,4012.2111,-0.0011,0.0246
9,20261106,68.3340,7836.3308,3975.7700,11812.1008,0.0198,0.0217



MODEL WIN COUNTS BY WEIGHTING SCHEME


,m1_wins,mean_delta,median_delta
scheme,,,
cap_100,8,0.00789,0.02130
cap_1000,8,0.00161,0.01370
equal_workload,9,0.01946,0.02430
execution,8,0.00450,0.00981



RANKING REVERSALS VS ORIGINAL EXECUTION WEIGHTING
equal_workload: [20261102]
cap_100: []
cap_1000: []


In [ ]:

import json
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/ASI_Experiment3/"
    "runs/exp3_restricted_v1/seeds"
)

for seed in [20261101, 20261102, 20261106]:
    print(f"\n{'=' * 65}")
    print(f"SEED {seed}")
    print(f"{'=' * 65}")

    for model in ["m0", "m1"]:
        path = BASE / str(seed) / model / "SELECTION" / "result.json"

        with open(path) as f:
            data = json.load(f)

        print(f"\n{model.upper()} top-level keys:")
        print(list(data.keys()))

        print("\nSelected fields and candidate summaries:")
        for key, value in data.items():
            if isinstance(value, list):
                print(f"  {key}: list of {len(value)} items")
                if value and isinstance(value[0], dict):
                    print("    First item keys:", list(value[0].keys()))
            elif isinstance(value, dict):
                print(f"  {key}: dict keys = {list(value.keys())}")
            else:
                print(f"  {key}: {value}")



SEED 20261101

M0 top-level keys:
['all_candidates', 'encoding', 'features', 'selected', 'test_used']

Selected fields and candidate summaries:
  all_candidates: list of 12 items
    First item keys: ['best_iteration', 'candidate_id', 'parameters', 'prediction_seconds', 'resources_so_far', 'training_seconds', 'validation_log_mae', 'validation_metrics']
  encoding: dict keys = ['gpu_spec_public', 'is_genai_request', 'model_type_public', 'priority_class']
  features: list of 5 items
  selected: dict keys = ['best_iteration', 'candidate_id', 'parameters', 'prediction_seconds', 'resources_so_far', 'training_seconds', 'validation_log_mae', 'validation_metrics']
  test_used: False

M1 top-level keys:
['all_candidates', 'encoding', 'features', 'selected', 'test_used']

Selected fields and candidate summaries:
  all_candidates: list of 12 items
    First item keys: ['best_iteration', 'candidate_id', 'parameters', 'prediction_seconds', 'resources_so_far', 'training_seconds', 'validation_log_ma

In [ ]:

import json
import pandas as pd
from pathlib import Path

BASE = Path(
    "/content/drive/MyDrive/ASI_Experiment3/"
    "runs/exp3_restricted_v1/seeds"
)

records = []

for seed in [20261101, 20261102, 20261106]:
    for model in ["m0", "m1"]:
        path = BASE / str(seed) / model / "SELECTION" / "result.json"

        with open(path) as f:
            data = json.load(f)

        selected_id = data["selected"]["candidate_id"]

        for c in data["all_candidates"]:
            records.append({
                "seed": seed,
                "model": model.upper(),
                "candidate": c["candidate_id"],
                "rounds": c["best_iteration"],
                "validation_logmae": c["validation_log_mae"],
                "selected": c["candidate_id"] == selected_id,
                "parameters": c["parameters"]
            })

candidates = pd.DataFrame(records)

for seed in [20261101, 20261102, 20261106]:
    print(f"\n{'=' * 65}")
    print(f"SEED {seed}")
    print(f"{'=' * 65}")

    for model in ["M0", "M1"]:
        subset = candidates[
            (candidates.seed == seed) &
            (candidates.model == model)
        ].sort_values(["validation_logmae", "candidate"])

        print(f"\n{model} — candidates ranked by validation LogMAE")
        display(subset[
            ["candidate", "rounds", "validation_logmae", "selected"]
        ].round(6))

        print(
            "Rounds: min =",
            subset["rounds"].min(),
            "| median =",
            subset["rounds"].median(),
            "| max =",
            subset["rounds"].max()
        )

print("\nSELECTED CANDIDATE HYPERPARAMETERS")
display(
    candidates[candidates.selected][
        ["seed", "model", "candidate", "rounds", "parameters"]
    ]
)



SEED 20261101

M0 — candidates ranked by validation LogMAE


,candidate,rounds,validation_logmae,selected
1,2,36,0.723052,True
3,4,36,0.723052,False
0,1,45,0.739838,False
2,3,45,0.739838,False
5,6,23,0.759999,False
7,8,23,0.759999,False
4,5,32,0.762904,False
6,7,32,0.762904,False
9,10,12,0.766382,False
11,12,12,0.766382,False


Rounds: min = 3 | median = 27.5 | max = 45

M1 — candidates ranked by validation LogMAE


,candidate,rounds,validation_logmae,selected
13,2,1,0.767299,True
15,4,1,0.767299,False
21,10,2,0.769183,False
23,12,2,0.769183,False
17,6,1,0.770651,False
19,8,1,0.770651,False
12,1,5,0.772788,False
14,3,5,0.772788,False
22,11,2,0.775478,False
20,9,2,0.775489,False


Rounds: min = 1 | median = 1.5 | max = 5

SEED 20261102

M0 — candidates ranked by validation LogMAE


,candidate,rounds,validation_logmae,selected
25,2,23,0.801967,True
27,4,23,0.801967,False
24,1,19,0.816238,False
26,3,19,0.816238,False
29,6,12,0.822889,False
31,8,12,0.822889,False
28,5,11,0.825385,False
30,7,11,0.825385,False
35,12,13,0.829595,False
33,10,13,0.829614,False


... (4 outputs left out)

Rounds: min = 148 | median = 199.0 | max = 200

M1 — candidates ranked by validation LogMAE


,candidate,rounds,validation_logmae,selected
64,5,194,0.659762,True
68,9,199,0.659798,False
70,11,199,0.660176,False
69,10,169,0.660225,False
71,12,194,0.660227,False
66,7,199,0.660349,False
65,6,200,0.660463,False
67,8,199,0.660648,False
60,1,200,0.662852,False
62,3,200,0.662852,False


Rounds: min = 169 | median = 199.0 | max = 200

SELECTED CANDIDATE HYPERPARAMETERS


,seed,model,candidate,rounds,parameters
1,20261101,M0,2,36,"{'bagging_fraction': 0.8, 'bagging_freq': 1, '..."
13,20261101,M1,2,1,"{'bagging_fraction': 0.8, 'bagging_freq': 1, '..."
25,20261102,M0,2,23,"{'bagging_fraction': 0.8, 'bagging_freq': 1, '..."
37,20261102,M1,2,9,"{'bagging_fraction': 0.8, 'bagging_freq': 1, '..."
57,20261106,M0,10,200,"{'bagging_fraction': 0.8, 'bagging_freq': 1, '..."
64,20261106,M1,5,194,"{'bagging_fraction': 0.8, 'bagging_freq': 1, '..."


In [ ]:

import numpy as np
import pandas as pd

rows = []
groups = []

for seed in seeds:
    df = tables[seed].copy()
    n = df["executions"].to_numpy(dtype=float)

    df["delta_per_exec"] = (
        df["m0_loss_sum"] - df["m1_loss_sum"]
    ) / df["executions"]

    df["delta_sum"] = (
        df["m0_loss_sum"] - df["m1_loss_sum"]
    )

    df["m1_better"] = df["delta_per_exec"] > 0

    rows.append({
        "seed": seed,
        "workloads": len(df),
        "executions": int(n.sum()),
        "pct_workloads_m1_better":
            100 * df["m1_better"].mean(),
        "pct_executions_on_m1_better_workloads":
            100 * df.loc[df.m1_better, "executions"].sum()
            / n.sum(),
        "median_workload_delta":
            df["delta_per_exec"].median(),
        "mean_workload_delta":
            df["delta_per_exec"].mean(),
        "execution_delta":
            df["delta_sum"].sum() / n.sum(),
        "p90_execution_count":
            df["executions"].quantile(.90),
        "p99_execution_count":
            df["executions"].quantile(.99),
        "max_execution_count":
            df["executions"].max()
    })

    # Frequency bands are defined without using prediction errors.
    df["frequency_band"] = pd.cut(
        df["executions"],
        bins=[0, 1, 10, 100, 1000, np.inf],
        labels=["1", "2-10", "11-100", "101-1000", ">1000"]
    )

    for band, g in df.groupby(
        "frequency_band", observed=True
    ):
        groups.append({
            "seed": seed,
            "band": str(band),
            "workloads": len(g),
            "execution_share_pct":
                100 * g["executions"].sum() / n.sum(),
            "pct_workloads_m1_better":
                100 * g["m1_better"].mean(),
            "delta_per_execution":
                g["delta_sum"].sum() / g["executions"].sum(),
            "contribution_to_global_delta":
                g["delta_sum"].sum() / n.sum()
        })

summary = pd.DataFrame(rows)
bands = pd.DataFrame(groups)

print("ALL TEN SEEDS — WORKLOAD ERROR COMPOSITION")
display(summary.round(5))

print("\nFREQUENCY-BAND DECOMPOSITION")
display(
    bands[bands.seed.isin(
        [20261101, 20261102, 20261106]
    )].round(5)
)

print("\nM1-BETTER WORKLOAD SHARE ACROSS SEEDS")
display(
    summary[[
        "seed",
        "pct_workloads_m1_better",
        "pct_executions_on_m1_better_workloads",
        "mean_workload_delta",
        "execution_delta"
    ]].round(5)
)

# Sanity checks against the original aggregate evaluation.
for seed in seeds:
    df = tables[seed]
    original = sensitivity[
        (sensitivity.seed == seed) &
        (sensitivity.scheme == "execution")
    ]["delta"].iloc[0]
    recomputed = (
        (df["m0_loss_sum"] - df["m1_loss_sum"]).sum()
        / df["executions"].sum()
    )
    assert np.isclose(original, recomputed, atol=1e-9)

print("\nAll aggregate consistency checks passed.")


ALL TEN SEEDS — WORKLOAD ERROR COMPOSITION


,seed,workloads,executions,pct_workloads_m1_better,pct_executions_on_m1_better_workloads,median_workload_delta,mean_workload_delta,execution_delta,p90_execution_count,p99_execution_count,max_execution_count
0,20261028,51189,498078,50.53234,56.93727,0.00045,0.01763,0.01904,5.0,152.00,20945
1,20261029,51328,513432,53.04512,45.40192,0.00185,0.02919,0.00725,4.0,156.00,17267
2,20261030,51404,497218,52.63793,43.76833,0.00078,0.02618,0.01377,5.0,149.97,17267
3,20261031,50961,476885,51.87496,52.79134,0.00155,0.03288,0.01007,5.0,149.00,9025
4,20261101,50814,551211,45.24344,54.80660,-0.01098,-0.01956,-0.04425,5.0,151.00,38159
5,20261102,50999,553132,53.82655,42.91670,0.00735,0.00785,-0.01616,5.0,153.02,38159
6,20261103,51287,425122,55.18747,56.15894,0.01170,0.02714,0.01713,4.0,145.00,12066
7,20261104,51611,532705,51.18289,51.11797,0.00030,0.02236,0.00955,5.0,152.00,22903
8,20261105,51380,491615,53.89646,45.27791,0.00321,0.02855,0.00816,5.0,143.21,15998
9,20261106,51145,578665,52.23384,56.25794,0.00557,0.02242,0.02041,5.0,154.00,30758



FREQUENCY-BAND DECOMPOSITION


,seed,band,workloads,execution_share_pct,pct_workloads_m1_better,delta_per_execution,contribution_to_global_delta
20,20261101,1,37454,6.79486,45.06595,0.00310,0.00021
21,20261101,2-10,10538,7.08749,47.05827,-0.07769,-0.00551
22,20261101,11-100,2036,12.25465,41.94499,-0.06839,-0.00838
23,20261101,101-1000,722,33.36526,35.45706,-0.14376,-0.04797
24,20261101,>1000,64,40.49774,65.62500,0.04296,0.01740
25,20261102,1,37640,6.80489,56.44261,0.01241,0.00084
26,20261102,2-10,10555,7.07679,48.18569,-0.00798,-0.00057
27,20261102,11-100,2030,12.40084,41.87192,-0.01100,-0.00136
28,20261102,101-1000,705,33.44771,34.32624,-0.03964,-0.01326
29,20261102,>1000,69,40.26977,40.57971,-0.00452,-0.00182



M1-BETTER WORKLOAD SHARE ACROSS SEEDS


,seed,pct_workloads_m1_better,pct_executions_on_m1_better_workloads,mean_workload_delta,execution_delta
0,20261028,50.53234,56.93727,0.01763,0.01904
1,20261029,53.04512,45.40192,0.02919,0.00725
2,20261030,52.63793,43.76833,0.02618,0.01377
3,20261031,51.87496,52.79134,0.03288,0.01007
4,20261101,45.24344,54.80660,-0.01956,-0.04425
5,20261102,53.82655,42.91670,0.00785,-0.01616
6,20261103,55.18747,56.15894,0.02714,0.01713
7,20261104,51.18289,51.11797,0.02236,0.00955
8,20261105,53.89646,45.27791,0.02855,0.00816
9,20261106,52.23384,56.25794,0.02242,0.02041



All aggregate consistency checks passed.
